In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# EMBEDDING BASICS: Learnable lookup table
# ============================================================

vocab_size = 10000   # Number of unique tokens
embed_dim = 256      # Embedding vector size

# Create embedding layer (stores a [10000, 256] matrix)
embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

print(f"Embedding matrix shape: {embedding.weight.shape}")
print(f"Total parameters: {embedding.weight.numel():,}")

# ============================================================
# LOOKUP: Token index -> dense vector
# ============================================================

# Simulate token indices for "The cat sat on the mat"
token_ids = torch.tensor([42, 1587, 923, 15, 42, 2041])

# Lookup: each index selects a row from the matrix
vectors = embedding(token_ids)
print(f"\nInput shape:  {token_ids.shape}")   # [6]
print(f"Output shape: {vectors.shape}")        # [6, 256]

# Same token (42 = "the") always gets the same vector
print(f"\nSame token, same vector: {torch.equal(vectors[0], vectors[4])}")

# ============================================================
# WHY IT WORKS: Equivalent to one-hot x matrix
# ============================================================

# One-hot approach (wasteful but mathematically identical)
one_hot = F.one_hot(torch.tensor(42), num_classes=vocab_size).float()
manual_lookup = one_hot @ embedding.weight  # Matrix multiply

# Direct indexing (what PyTorch actually does)
direct_lookup = embedding.weight[42]

print(f"\nOne-hot multiply == direct index: "
      f"{torch.allclose(manual_lookup, direct_lookup)}")

# ============================================================
# SIMILARITY: Trained embeddings cluster similar words
# ============================================================

# After training, similar words have high cosine similarity
v1 = vectors[0]  # "the"
v2 = vectors[1]  # "cat"
cosine_sim = F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0))
print(f"\nCosine similarity (untrained, random): {cosine_sim.item():.4f}")
print("After training, similar words would score > 0.7")

Embedding matrix shape: torch.Size([10000, 256])
Total parameters: 2,560,000

Input shape:  torch.Size([6])
Output shape: torch.Size([6, 256])

Same token, same vector: True

One-hot multiply == direct index: True

Cosine similarity (untrained, random): -0.1796
After training, similar words would score > 0.7
